In [1]:
#PREPROCESSING
import pandas as pd
from pathlib import Path
from sklearn.utils import shuffle


DATA_DIR = Path("/home/vinod.bareeti/roberta_data")

data_csv = DATA_DIR / "data.csv"
train_csv = DATA_DIR / "train_v2_drcat_02.csv"
out_csv = DATA_DIR / "combined_dataset.csv"

print("Using files:")
print(" data.csv:", data_csv.exists())
print(" train_v2_drcat_02.csv:", train_csv.exists())

df_data = pd.read_csv(data_csv)
df_train = pd.read_csv(train_csv)

print("\nLoaded shapes:")
print(" df_data :", df_data.shape)
print(" df_train:", df_train.shape)

def map_data_label(src):
    s = str(src).strip().lower()
    if s == "human":
        return "human"
    ai_keywords = ["gpt", "bloom", "llama", "ai", "model"]
    if any(k in s for k in ai_keywords):
        return "ai"
    return None

df_data["label"] = df_data["source"].apply(map_data_label)

print("\nMapping counts in data.csv:")
print(df_data["label"].value_counts(dropna=False))

df_data = df_data[df_data["label"].notna()].reset_index(drop=True)
df_data_prepared = df_data[["text", "label"]].copy()

df_train_prepared = pd.DataFrame()
df_train_prepared["text"] = df_train["text"].astype(str)
df_train_prepared["label"] = df_train["label"].apply(
    lambda x: "ai" if int(x) == 1 else "human"
)

print("\nLabel counts in train file:")
print(df_train_prepared["label"].value_counts())

combined = pd.concat([df_train_prepared, df_data_prepared], ignore_index=True)
combined = shuffle(combined, random_state=42).reset_index(drop=True)

print("\nFINAL Combined shape:", combined.shape)
print("Final label distribution:")
print(combined["label"].value_counts())

combined.to_csv(out_csv, index=False)

print("\nCombined dataset saved to:", out_csv)

combined.head(10)


Using files:
 data.csv: True
 train_v2_drcat_02.csv: True

Loaded shapes:
 df_data : (788922, 5)
 df_train: (44868, 5)

Mapping counts in data.csv:
label
human    347692
None     299534
ai       141696
Name: count, dtype: int64

Label counts in train file:
label
human    27371
ai       17497
Name: count, dtype: int64

FINAL Combined shape: (534256, 2)
Final label distribution:
label
human    375063
ai       159193
Name: count, dtype: int64

Combined dataset saved to: /home/vinod.bareeti/roberta_data/combined_dataset.csv


,text,label
0,Counselors as Social Justice Advocates Report ...,human
1,Just go to Tokyo Lobby on 59th and bell. Bette...,human
2,"When searching for advice, do you ask more tha...",human
3,"When your mouth makes the sound of ""p,"" it als...",human
4,Group Knowledge Sharing in Technology Manageme...,human
5,Because bread does not stale just when moistur...,human
6,Jack and Tom were friends for more than fiftee...,ai
7,The Study of the Constituents of the Brain Ess...,human
8,"Fear: Definition, Effects, and Overcoming Essa...",human
9,"In ""The challenge of Exploring Venus"" the auth...",human


In [2]:
#Imports & Setup
import pandas as pd

df = pd.read_csv("/home/vinod.bareeti/roberta_data/combined_dataset.csv")

# encode labels
df["label"] = df["label"].map({"human": 0, "ai": 1})

print(df.shape)
df["label"].value_counts()


(534256, 2)


label
0    375063
1    159193
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df.shape, test_df.shape


((427404, 2), (106852, 2))

In [4]:
train_small = train_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(20000, random_state=42)
)

test_small = test_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(5000, random_state=42)
)

train_small["label"].value_counts(), test_small["label"].value_counts()


/tmp/ipykernel_106613/2016719364.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_small = train_df.groupby("label", group_keys=False).apply(
/tmp/ipykernel_106613/2016719364.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_small = test_df.groupby("label", group_keys=False).apply(


(label
 0    20000
 1    20000
 Name: count, dtype: int64,
 label
 0    5000
 1    5000
 Name: count, dtype: int64)

In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_small.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_small.reset_index(drop=True))

train_dataset, test_dataset


(Dataset({
     features: ['text', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 10000
 }))

In [6]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
print("Starting tokenization for TRAIN dataset...")
train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    desc="Tokenizing TRAIN"
)
print("Train tokenization done")

print("Starting tokenization for TEST dataset...")
test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    desc="Tokenizing TEST"
)
print("Test tokenization done")


Starting tokenization for TRAIN dataset...


Tokenizing TRAIN:   0%|          | 0/40000 [00:00<?, ? examples/s]

✅ Train tokenization done
Starting tokenization for TEST dataset...


Tokenizing TEST:   0%|          | 0/10000 [00:00<?, ? examples/s]

✅ Test tokenization done


In [8]:
print(train_dataset.features)


{'text': Value('string'), 'label': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [9]:
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 40000
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 10000
})


In [11]:
import torch
from transformers import RobertaForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

model.to(device)


Using device: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size = 16
    num_train_epochs=15,             # ✅ 15 epochs
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,              # keeps disk usage low
    report_to=[]
)



In [16]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


In [18]:
from transformers import EarlyStoppingCallback

trainer.add_callback(
    EarlyStoppingCallback(early_stopping_patience=3)
)


In [19]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.148200,0.304122,0.913600,0.860782,0.986800,0.919493
2,0.074200,0.345139,0.933600,0.889788,0.989800,0.937133
3,0.072300,0.194606,0.958700,0.933636,0.987600,0.959860
4,0.047300,0.654642,0.897700,0.834033,0.993000,0.906601
5,0.042500,0.949672,0.879500,0.809291,0.993000,0.891783
6,0.025400,0.467348,0.930100,0.883538,0.990800,0.934100


TrainOutput(global_step=15000, training_loss=0.07190368523796399, metrics={'train_runtime': 3169.6664, 'train_samples_per_second': 189.294, 'train_steps_per_second': 11.831, 'total_flos': 3.15733266432e+16, 'train_loss': 0.07190368523796399, 'epoch': 6.0})

In [20]:
trainer.evaluate()


{'eval_loss': 0.19460631906986237,
 'eval_accuracy': 0.9587,
 'eval_precision': 0.9336358479863869,
 'eval_recall': 0.9876,
 'eval_f1': 0.9598600447079405,
 'eval_runtime': 38.4327,
 'eval_samples_per_second': 260.195,
 'eval_steps_per_second': 65.049,
 'epoch': 6.0}

In [21]:
model.eval()


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [33]:
import torch

def predict_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
    
    label = "AI-generated" if pred == 1 else "Human-written"
    confidence = probs[0][pred].item()
    
    return label, confidence


In [48]:
trainer.save_model("./roberta_ai_human_detector")


In [82]:
text = "I am \\nolsdn;oilkfdhjof khn sd lhjbwvs doilk isbpohqwgfejgp wfeiklqvbfoigbw uqj dgouhjwqd avuowyiladj vabiyohlbkqcvuhjq kguvdch jkgwvdiuoqvd hcj"
label, confidence = predict_text(text)

print("Prediction:", label)
print("Confidence:", round(confidence * 100, 2), "%")

Prediction: Human-written
Confidence: 85.91 %
